# Qwen2.5-VL SFT Fine-Tuning (Single T4 / 2xT4 on Kaggle)

This notebook runs Supervised Fine-Tuning (SFT) on Qwen2.5-VL-3B using PRM-verified correct reasoning trajectories.
**Goal**: Enforce structured step-by-step outputs, improve logical reasoning, and reduce visual/arithmetic hallucinations.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret('HF_TOKEN')
    print('Successfully loaded HF_TOKEN secrets from Kaggle.')
except Exception as e:
    print('Kaggle secrets not available or skipped.')

!git clone https://github.com/yahorlahunovich/prm_project.git
%cd prm_project

In [ ]:
!pip install -qU transformers accelerate peft bitsandbytes trl datasets
!pip install -q qwen-vl-utils

In [ ]:
import logging
import datasets
import transformers
datasets.logging.set_verbosity_info()
transformers.logging.set_verbosity_info()

import json
import os
import torch
from datasets import Dataset
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from PIL import Image

# 1. Ensure chart images directory exists & download images if missing
images_dir = 'data/CharXiv/images'
if not os.path.exists(images_dir) or len(os.listdir(images_dir)) == 0:
    print('Images directory empty or missing. Auto-downloading chart images...')
    os.system('python scripts/download_images.py')

# 2. Extract verified correct reasoning trajectories for SFT
evals_path = 'experiments/001_500_reasoning/data/evaluated_rollouts.jsonl'
cleaned_path = 'experiments/001_500_reasoning/data/001_500_reasoning_cleaned.jsonl'

meta = {}
with open(cleaned_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        qid = str(data['question_id'])
        ridx = data['rollout_index']
        gt = str(data['ground_truth']).strip().lower()
        ans = str(data['model_final_answer']).strip().lower()
        is_correct = (gt in ans or ans in gt) and len(ans) > 0
        meta[(qid, ridx)] = {
            'is_correct': is_correct,
            'question': data.get('question', ''),
            'reasoning': data.get('reasoning_steps', '')
        }

sft_conversations = []
with open(evals_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        qid = str(data['question_id'])
        ridx = data['rollout_index']
        evals = data.get('evaluations', [])
        if not evals:
            continue
        all_pass = all(s.get('score') == 1 for s in evals)
        m = meta.get((qid, ridx), {})
        if all_pass and m.get('is_correct', False) and m.get('reasoning'):
            abs_path = os.path.abspath(f'data/CharXiv/images/{qid}.jpg')
            if not os.path.exists(abs_path):
                continue
            
            # Re-save image cleanly as RGB JPEG to prevent torchvision header errors
            try:
                img = Image.open(abs_path).convert('RGB')
                img.save(abs_path, format='JPEG', quality=95)
            except Exception as e:
                continue

            prompt_text = 'Analyze this chart. Provide step-by-step reasoning and a final answer.
' + m['question']
            
            messages = [
                {
                    'role': 'user',
                    'content': [
                        {'type': 'image', 'image': abs_path},
                        {'type': 'text', 'text': prompt_text}
                    ]
                },
                {
                    'role': 'assistant',
                    'content': m['reasoning']
                }
            ]
            sft_conversations.append({'messages': messages})

print(f'Extracted {len(sft_conversations)} verified positive SFT reasoning trajectories.')
dataset = Dataset.from_list(sft_conversations)
print(dataset)

In [ ]:
# 3. Load Model & Processor
model_id = 'Qwen/Qwen2.5-VL-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    attn_implementation='sdpa',
    device_map={'': 0}
)

model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

if hasattr(model, 'visual'):
    model.visual.requires_grad_(False)

processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=512*28*28)
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

In [ ]:
# 4. Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    task_type='CAUSAL_LM',
)

In [ ]:
# 5. Define SFT Trainer & Train
training_args = SFTConfig(
    output_dir='./sft_qwen_vl',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=1,
    save_steps=10,
    gradient_checkpointing=True,
    dataset_num_proc=1,
    remove_unused_columns=False,
    report_to='none',
    dataset_text_field='messages',
)

print('
=== INITIALIZING SFTTRAINER ===')
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=processor,
    peft_config=peft_config,
)

print('
=== STARTING SFT TRAINING LOOP ===')
trainer.train()
trainer.save_model('qwen_vl_sft_adapter')
print('
SFT Training complete! Adapter saved to qwen_vl_sft_adapter.')